In [2]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

True

In [3]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = "telecom_guide.pdf"

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

/var/folders/gf/9w9vq_wn57n49drhh3ck7kbm0000gn/T/ipykernel_57042/2523830466.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [4]:
print(f"Loaded {len(pages)} pages from pdf")
print("=== FIRST PAGE PREVIEW ===")
print(pages[0].page_content[:500])

Loaded 9 pages from pdf
=== FIRST PAGE PREVIEW ===
Telecom Technical Reference Guide  - Internal Use Only
Telecom Technical
Reference Guide
Customer Care & Network Operations Edition
Version 3.2  |  Covers 2G / 3G / 4G LTE / 5G
Page 1


### Create Chuns 

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100,
    separators=["\n\n","\n","."," "]
)

chunks = splitter.split_documents(pages)
len(chunks)

37

In [6]:
print(chunks[2].page_content)

downloads at speeds of 1-14 Mbps. This generation established the foundation for smartphone adoption
worldwide.
4G (LTE) networks delivered a major leap forward, with typical download speeds of 20-150 Mbps and latency
below 50 ms. LTE networks are fully IP-based, meaning voice calls are transmitted as data (VoLTE) rather than
over a separate circuit-switched channel. LTE-Advanced (LTE-A) uses carrier aggregation to combine multiple
frequency bands, pushing speeds beyond 300 Mbps.
5G networks represent the current frontier. Deploying across sub-6 GHz and millimetre-wave (mmWave)


In [7]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"batch_size": 1},
)

v = embeddings.embed_query("test")
print(len(v)) 


Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 10887.71it/s]


384


In [8]:
print(len(embeddings.embed_query("What is VoLTE?")))

384


In [9]:
from langchain_chroma import Chroma
vector_store = Chroma.from_documents(chunks,embeddings)

print(f"Vector store ready. {vector_store._collection.count()} vectors stored")

Vector store ready. 37 vectors stored


In [10]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

test_query = "What is volte and how does it improve code quality" 
retrieved  = retriever.invoke(test_query)

In [15]:
retrieved

[Document(id='70e13493-69aa-4634-b7ba-6424a3bb1c90', metadata={'page': 6, 'creator': 'PyPDF', 'page_label': '7', 'creationdate': '2026-04-30T14:16:24+00:00', 'source': 'telecom_guide.pdf', 'producer': 'PyPDF', 'total_pages': 9}, page_content='Telecom Technical Reference Guide  - Internal Use Only\n6. VoLTE, VoWiFi, and Advanced Voice Services\nVoice over LTE (VoLTE) and Voice over Wi-Fi (VoWiFi) are IP-based voice technologies that replace the legacy\ncircuit-switched voice channel used in 2G and 3G networks.\nVoLTE: With VoLTE, voice calls are transmitted as data packets over the LTE network using the IMS (IP\nMultimedia Subsystem) core. Benefits include HD voice quality (wideband audio at 16 kHz versus the 3.4 kHz of\nlegacy calls), faster call setup times (under 2 seconds versus 6-8 seconds on 3G), and the ability to use data and'),
 Document(id='f14fa130-2ca3-4353-89ae-2ac62b26454e', metadata={'page_label': '7', 'total_pages': 9, 'creator': 'PyPDF', 'source': 'telecom_guide.pdf', '

In [14]:
for i,doc in enumerate(retrieved,1):
    print(f"--- Chunk {i} ---")
    print(doc.page_content[:300])
    print()

--- Chunk 1 ---
Telecom Technical Reference Guide  - Internal Use Only
6. VoLTE, VoWiFi, and Advanced Voice Services
Voice over LTE (VoLTE) and Voice over Wi-Fi (VoWiFi) are IP-based voice technologies that replace the legacy
circuit-switched voice channel used in 2G and 3G networks.
VoLTE: With VoLTE, voice calls 

--- Chunk 2 ---
voice simultaneously without degradation. VoLTE requires a compatible device, a VoLTE-enabled SIM, and an
account that has VoLTE activated.
Enabling VoLTE: On most Android devices navigate to Settings > Mobile Network > VoLTE and toggle it on. On
iPhone go to Settings > Mobile Data > Mobile Data Opt

--- Chunk 3 ---
prioritised over general data traffic. This prevents voice quality degradation during periods of network congestion.
Without QoS, voice packets would compete with video streaming and file downloads, causing jitter and packet
loss.
Fallback Behaviour: If a VoLTE call cannot be established  - for exam



### SYS PROMPT

In [26]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq

# --- Helper to join retrived chunks into a single context string
def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)
    

    
SYSTEM_PROMT  = """\
You are a helpfull telecom assistant.
Answer the questionusing ONLY the context provided below.
If the context does not contain enough information, Say so clearly.

Context :
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system",SYSTEM_PROMT),
    ("human","{question}")
])


llm = ChatGroq(
    model="qwen/qwen3.8-27b",
    temperature=0,
    reasoning_format="parsed"
)

chain = (
    { "context" : retriever | format_docs,"question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG Chain Assembled")

RAG Chain Assembled


In [27]:
question = "How does international roaming works and what charges I should expect"
print(f"Q: {question} \n")

print(f"A: {chain.invoke(question)}")

Q: How does international roaming works and what charges I should expect 

A: Based on the provided context, here is how international roaming works and the charges you should expect:

**How It Works**
*   **Connection:** When you travel outside your home network's coverage area, your device connects to a partner network in the visited country.
*   **Authentication & Billing:** The visited network authenticates you via inter-operator signalling (SS7 or Diameter). Your home network validates your subscription and authorizes service. All voice, data, and SMS traffic is tunnelled back to the home network for billing.
*   **Latency:** This tunnelling process adds latency compared to using a local network.

**Charges and Roaming Zones**
Charges depend on which "Roaming Zone" the country you are visiting falls into:
*   **Zone A (EU, UK, Australia, New Zealand):** Lowest roaming rates.
*   **Zone B (USA, Canada, Japan, Singapore):** Moderate rates.
*   **Zone C (Rest of World):** Highest per